# 01 · Data check
Visual sanity check of `data/processed/`. Every crisis should show up where we expect it:
banks crash in 2008 and 2011, foreign assets drop in CHF in **January 2015**, everything falls in March 2020,
stocks **and** bonds fall in 2022, CHF rates are negative from 2015 to 2022.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# find the project folder (works if the notebook runs from notebooks/ or from the project root)
ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"

returns = pd.read_csv(PROC / "returns_chf.csv", index_col="date")
rates   = pd.read_csv(PROC / "rates.csv", index_col="date")
returns.index = pd.PeriodIndex(returns.index, freq="M").to_timestamp("M")
rates.index   = pd.PeriodIndex(rates.index, freq="M").to_timestamp("M")

CRISES = {
    "2008 GFC":        ("2007-11", "2009-02"),
    "2011 Euro crisis": ("2011-05", "2011-09"),
    "Jan 2015 CHF shock": ("2015-01", "2015-01"),
    "2020 COVID":      ("2020-02", "2020-03"),
    "2022 rates shock": ("2022-01", "2022-09"),
}
print(returns.shape)
returns.tail()

## 1. Coverage and missing values

In [ ]:
cov = pd.DataFrame({
    "first": returns.apply(lambda s: s.first_valid_index()).dt.strftime("%Y-%m"),
    "last":  returns.apply(lambda s: s.last_valid_index()).dt.strftime("%Y-%m"),
    "months": returns.notna().sum(),
    "missing inside range": returns.apply(lambda s: s.loc[s.first_valid_index():s.last_valid_index()].isna().sum()),
})
cov   # 'missing inside range' should be 0 everywhere

## 2. Summary statistics (annualised)

In [ ]:
stats = pd.DataFrame({
    "mean": returns.mean() * 12,
    "vol": returns.std() * np.sqrt(12),
    "worst month": returns.min(),
    "best month": returns.max(),
})
stats["sharpe-ish"] = (returns.sub(rates["rf_chf"], axis=0).mean() * 12) / stats["vol"]
stats.style.format("{:.1%}", subset=["mean", "vol", "worst month", "best month"]).format("{:.2f}", subset=["sharpe-ish"])

## 3. Cumulative returns in CHF, crises shaded

In [ ]:
def shade_crises(ax):
    for name, (a, b) in CRISES.items():
        ax.axvspan(pd.Period(a, "M").start_time, pd.Period(b, "M").end_time, color="grey", alpha=0.15)

groups = {
    "Equities":           ["world_eq", "smi", "eu_banks", "ubs"],
    "Bonds":              ["chf_bonds", "global_bonds", "global_bonds_hedged"],
    "Real assets":        ["swiss_re", "gold"],
    "Hedged vs unhedged": ["world_eq", "world_eq_hedged", "eu_banks", "eu_banks_hedged"],
}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (title, cols) in zip(axes.ravel(), groups.items()):
    start = "2001-10"  # common start so lines are comparable
    wealth = (1 + returns.loc[start:, cols].fillna(0)).cumprod()
    wealth.plot(ax=ax, logy=True)
    shade_crises(ax)
    ax.set_title(title); ax.set_ylabel("growth of CHF 1 (log)"); ax.set_xlabel("")
fig.tight_layout()

## 4. What happened in each crisis (total return in CHF)

In [ ]:
crisis_table = pd.DataFrame({
    name: (1 + returns.loc[a:b]).prod() - 1 for name, (a, b) in CRISES.items()
}).T
crisis_table.style.format("{:.1%}").background_gradient(cmap="RdYlGn", vmin=-0.5, vmax=0.5)

**What to check:** banks and UBS should be the worst in 2008 (around −50% or worse); in Jan 2015 the
foreign *unhedged* assets should lose roughly 10–15% in CHF while Swiss assets and *hedged* ones lose much less;
in 2022 both equities and bonds should be negative.

## 5. Correlations

In [ ]:
core = ["world_eq", "smi", "eu_banks", "ubs", "chf_bonds", "global_bonds_hedged", "swiss_re", "gold"]
corr = returns[core].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(core)), core, rotation=45, ha="right"); ax.set_yticks(range(len(core)), core)
for i in range(len(core)):
    for j in range(len(core)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im); ax.set_title("Monthly return correlations (CHF)"); fig.tight_layout()

**What to check:** eu_banks and ubs strongly correlated (they are both 'the banker's sector'),
equities positively correlated with each other, CHF bonds close to zero or negative with equities.

## 6. Risk-free rates

In [ ]:
ax = (rates * 12 * 100).plot(figsize=(12, 4))
ax.axhline(0, color="black", lw=0.8)
shade_crises(ax)
ax.set_ylabel("% per year"); ax.set_title("Monthly risk-free rates (annualised)")

**What to check:** `rf_chf` negative from about 2015 to 2022 (SNB at −0.75%).

## 7. Biggest single months (to spot remaining data errors)

In [ ]:
extremes = returns.stack().rename("return").reset_index()
extremes.columns = ["date", "asset", "return"]
extremes["abs"] = extremes["return"].abs()
extremes.sort_values("abs", ascending=False).head(15).drop(columns="abs")

Each of these should be explainable by a real event (2008, March 2020, the UBS rescue, …). If one is not, it is probably a data error.